# Customer Churn End-to-End Analysis
EDA, feature engineering, PyCaret classification, and parquet exports for Power BI.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.pipeline import (
    load_dataset,
    infer_target_column,
    basic_cleaning,
    feature_engineering,
    export_parquet_assets,
    build_powerbi_ready_tables,
    export_powerbi_model_tables,
)

## 1. Load Data
Set your raw dataset file path below.

In [3]:
raw_dir = project_root / "data" / "raw"
train_file = raw_dir / "customer_churn_dataset-training-master.csv"
test_file = raw_dir / "customer_churn_dataset-testing-master.csv"

df_train = load_dataset(train_file)
df_test = load_dataset(test_file)
df_raw = pd.concat([df_train, df_test], ignore_index=True)
print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print("Combined shape:", df_raw.shape)
df_raw.head()

Train shape: (440833, 12)
Test shape: (64374, 12)
Combined shape: (505207, 12)


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


## 2. EDA and Data Quality Checks

In [3]:
print(f"Shape: {df_raw.shape}")
display(df_raw.info())
display(df_raw.describe(include='all').T.head(20))
missing_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct[missing_pct > 0].head(20))

Shape: (505207, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 505207 entries, 0 to 505206
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         505206 non-null  float64
 1   Age                505206 non-null  float64
 2   Gender             505206 non-null  object 
 3   Tenure             505206 non-null  float64
 4   Usage Frequency    505206 non-null  float64
 5   Support Calls      505206 non-null  float64
 6   Payment Delay      505206 non-null  float64
 7   Subscription Type  505206 non-null  object 
 8   Contract Length    505206 non-null  object 
 9   Total Spend        505206 non-null  float64
 10  Last Interaction   505206 non-null  float64
 11  Churn              505206 non-null  float64
dtypes: float64(9), object(3)
memory usage: 46.3+ MB


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
CustomerID,505206.0,NaN,NaN,NaN,200779.451782,137241.343095,1.0,63827.25,193039.5,321645.75,449999.0
Age,505206.0,NaN,NaN,NaN,39.704172,12.670577,18.0,29.0,40.0,49.0,65.0
Gender,505206,2,Male,280273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tenure,505206.0,NaN,NaN,NaN,31.350435,17.237482,1.0,16.0,32.0,46.0,60.0
Usage Frequency,505206.0,NaN,NaN,NaN,15.714825,8.619323,1.0,8.0,16.0,23.0,30.0
Support Calls,505206.0,NaN,NaN,NaN,3.833317,3.133603,0.0,1.0,3.0,6.0,10.0
Payment Delay,505206.0,NaN,NaN,NaN,13.496843,8.451187,0.0,6.0,13.0,20.0,30.0
Subscription Type,505206,3,Standard,170630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Contract Length,505206,3,Annual,198608,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Total Spend,505206.0,NaN,NaN,NaN,620.072766,245.319256,100.0,446.0,648.9,824.0,1000.0


CustomerID           0.000198
Age                  0.000198
Gender               0.000198
Tenure               0.000198
Usage Frequency      0.000198
Support Calls        0.000198
Payment Delay        0.000198
Subscription Type    0.000198
Contract Length      0.000198
Total Spend          0.000198
Last Interaction     0.000198
Churn                0.000198
dtype: float64

In [4]:
df_clean = basic_cleaning(df_raw)
target_col = infer_target_column(df_clean)
print(f"Target column inferred: {target_col}")
df_clean.head()

Target column inferred: Churn


,CustomerID,Age,Gender,Tenure,Usage_Frequency,Support_Calls,Payment_Delay,Subscription_Type,Contract_Length,Total_Spend,Last_Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


## 3. Quick Visual EDA

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_clean, x=target_col)
plt.title("Target Distribution")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [5]:
import importlib
import src.data.pipeline as pipeline_module

pipeline_module = importlib.reload(pipeline_module)
X, y = pipeline_module.feature_engineering(df_clean, target_col=target_col)
print(f"Model feature count: {X.shape[1]}")

Model feature count: 12


## 5. PyCaret Classification (Multiple Model Analysis)

In [6]:
from pycaret.classification import setup, compare_models, pull, evaluate_model

model_df = pd.concat([X, y], axis=1).copy()

# Use stratified sampling for faster experimentation on large datasets.
sample_size = 60000
if len(model_df) > sample_size:
    model_df = (
        model_df.groupby(target_col, group_keys=False)
        .apply(lambda g: g.sample(max(1, int(sample_size * len(g) / len(pd.concat([X, y], axis=1)))), random_state=42))
        .reset_index(drop=True)
    )

print("Modeling rows:", len(model_df))

exp = setup(
    data=model_df,
    target=target_col,
    session_id=42,
    train_size=0.8,
    fold=5,
    normalize=True,
    remove_multicollinearity=True,
    multicollinearity_threshold=0.9,
    verbose=False
)

best_models = compare_models(n_select=5, sort='AUC')
leaderboard = pull()
leaderboard.head(10)

Modeling rows: 59999


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.9334,0.9544,0.9944,0.8968,0.9431,0.8633,0.8700,0.2280
rf,Random Forest Classifier,0.9309,0.9521,0.9895,0.8967,0.9408,0.8582,0.8642,0.6040
gbc,Gradient Boosting Classifier,0.9181,0.9502,0.9654,0.8953,0.9290,0.8325,0.8358,0.9540
et,Extra Trees Classifier,0.9188,0.9477,0.9641,0.8973,0.9295,0.8341,0.8371,0.5540
ada,Ada Boost Classifier,0.8644,0.9382,0.8463,0.9035,0.8739,0.7277,0.7295,0.3440
knn,K Neighbors Classifier,0.8828,0.9241,0.8859,0.9013,0.8935,0.7632,0.7633,1.3160
nb,Naive Bayes,0.8530,0.9235,0.8209,0.9055,0.8611,0.7058,0.7096,0.3820
lr,Logistic Regression,0.8495,0.9089,0.8488,0.8762,0.8623,0.6964,0.6969,0.6720
ridge,Ridge Classifier,0.8428,0.9084,0.8233,0.8855,0.8533,0.6845,0.6866,0.0720
lda,Linear Discriminant Analysis,0.8427,0.9084,0.8230,0.8856,0.8532,0.6843,0.6864,0.0740


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.9334,0.9544,0.9944,0.8968,0.9431,0.8633,0.8700,0.228
rf,Random Forest Classifier,0.9309,0.9521,0.9895,0.8967,0.9408,0.8582,0.8642,0.604
gbc,Gradient Boosting Classifier,0.9181,0.9502,0.9654,0.8953,0.9290,0.8325,0.8358,0.954
et,Extra Trees Classifier,0.9188,0.9477,0.9641,0.8973,0.9295,0.8341,0.8371,0.554
ada,Ada Boost Classifier,0.8644,0.9382,0.8463,0.9035,0.8739,0.7277,0.7295,0.344
knn,K Neighbors Classifier,0.8828,0.9241,0.8859,0.9013,0.8935,0.7632,0.7633,1.316
nb,Naive Bayes,0.8530,0.9235,0.8209,0.9055,0.8611,0.7058,0.7096,0.382
lr,Logistic Regression,0.8495,0.9089,0.8488,0.8762,0.8623,0.6964,0.6969,0.672
ridge,Ridge Classifier,0.8428,0.9084,0.8233,0.8855,0.8533,0.6845,0.6866,0.072
lda,Linear Discriminant Analysis,0.8427,0.9084,0.8230,0.8856,0.8532,0.6843,0.6864,0.074


In [ ]:
# Optional: open interactive diagnostics for the top model
evaluate_model(best_models[0])

In [7]:
# ── Section 5b: Extract & Export Feature Importance ──────────────────────────
from pycaret.classification import get_config

X_train = get_config('X_train')

# Unwrap PyCaret pipeline to reach the actual estimator
best_model = best_models[0]
try:
    estimator = best_model.named_steps['actual_estimator']
except (AttributeError, KeyError):
    estimator = best_model

# Get raw importances
if hasattr(estimator, 'feature_importances_'):
    importances = estimator.feature_importances_
elif hasattr(estimator, 'coef_'):
    importances = abs(estimator.coef_[0])
else:
    importances = None

n = len(importances) if importances is not None else len(X_train.columns)

if hasattr(estimator, 'feature_names_in_') and len(estimator.feature_names_in_) == n:
    feature_names = list(estimator.feature_names_in_)
elif len(X_train.columns) == n:
    feature_names = list(X_train.columns)
else:
    try:
        preprocessor = best_model[:-1]
        X_tf = preprocessor.transform(X_train)
        if hasattr(X_tf, 'columns') and len(X_tf.columns) == n:
            feature_names = list(X_tf.columns)
        elif hasattr(preprocessor, 'get_feature_names_out'):
            feature_names = list(preprocessor.get_feature_names_out())[:n]
        else:
            feature_names = [f"feature_{i:02d}" for i in range(n)]
    except Exception:
        feature_names = [f"feature_{i:02d}" for i in range(n)]

if importances is None:
    importances = [1.0 / n] * n

dim_feature_importance = (
    pd.DataFrame({'feature': feature_names, 'importance': importances})
    .loc[lambda df: ~df['feature'].str.lower().str.replace('_', '').isin({'customerid', 'id'})]
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

out_path = project_root / "data" / "processed" / "parquet" / "dim_feature_importance.parquet"
dim_feature_importance.to_parquet(out_path, index=False)
print(f"Saved {len(dim_feature_importance)} features → {out_path}")
dim_feature_importance.head(10)

Saved 16 features → c:\Users\sayee\Documents\Customer Churn\data\processed\parquet\dim_feature_importance.parquet


,feature,importance
0,Total_Spend,467
1,Age,458
2,Payment_Delay,407
3,Tenure,375
4,Usage_Frequency,304
5,Support_Calls,292
6,Last_Interaction,292
7,Gender,180
8,Contract_Length_Monthly,105
9,Subscription_Type_Basic,28


## 6. Export Parquet for Power BI

In [14]:
export_parquet_assets(df_clean, X, y)
powerbi_tables = build_powerbi_ready_tables(df_clean, target_col=target_col)
export_powerbi_model_tables(powerbi_tables)
print("Parquet exports completed.")

Parquet exports completed.


## 6b. Export Explainability Tables
Create the model metrics, confusion matrix, ROC curve, precision-recall curve, threshold scan, and customer risk score tables needed by the Explainability and Risk pages in Power BI.

In [8]:
from pycaret.classification import get_config, predict_model
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

output_dir = project_root / "data" / "processed" / "parquet"
output_dir.mkdir(parents=True, exist_ok=True)


def _as_numeric_series(values, name):
    if isinstance(values, pd.DataFrame):
        if values.shape[1] != 1:
            raise ValueError(f"Expected a single-column object for {name}.")
        series = values.iloc[:, 0]
    elif isinstance(values, pd.Series):
        series = values
    else:
        series = pd.Series(values)
    return pd.to_numeric(series, errors="coerce").rename(name).reset_index(drop=True)


def _resolve_score_column(scored_df):
    preferred_columns = [
        "prediction_score_1",
        "prediction_score",
        "Score_1",
        "score_1",
        "Score",
        "score",
    ]
    for column in preferred_columns:
        if column in scored_df.columns:
            return column

    score_columns = [column for column in scored_df.columns if "score" in column.lower()]
    if score_columns:
        return score_columns[-1]

    raise KeyError("No score column was found in the PyCaret prediction output.")


X_test = get_config("X_test")
y_test = _as_numeric_series(get_config("y_test"), "actual_churn").fillna(0).astype(int)

test_scored = predict_model(best_model, data=X_test, raw_score=True).reset_index(drop=True)
test_score_column = _resolve_score_column(test_scored)
test_probability = pd.to_numeric(test_scored[test_score_column], errors="coerce").fillna(0.0).clip(0, 1)
test_predicted = (test_probability >= 0.50).astype(int)

fpr, tpr, roc_thresholds = roc_curve(y_test, test_probability)
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, test_probability)
pr_threshold_aligned = np.append(pr_thresholds, np.nan)

threshold_grid = np.round(np.linspace(0.05, 0.95, 19), 2)
threshold_rows = []
for threshold in threshold_grid:
    predicted_at_threshold = (test_probability >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": float(threshold),
            "precision": float(precision_score(y_test, predicted_at_threshold, zero_division=0)),
            "recall": float(recall_score(y_test, predicted_at_threshold, zero_division=0)),
            "f1": float(f1_score(y_test, predicted_at_threshold, zero_division=0)),
            "accuracy": float(accuracy_score(y_test, predicted_at_threshold)),
            "predicted_churners": int(predicted_at_threshold.sum()),
        }
    )

fact_threshold_metrics = pd.DataFrame(threshold_rows)
best_threshold_row = fact_threshold_metrics.sort_values(
    ["f1", "recall", "precision", "accuracy"],
    ascending=[False, False, False, False],
).iloc[0]

fact_confusion_matrix = pd.DataFrame(
    [
        {"actual_label": "Retained", "predicted_label": "Retained", "customer_count": int(confusion_matrix(y_test, test_predicted, labels=[0, 1])[0, 0])},
        {"actual_label": "Retained", "predicted_label": "Churned", "customer_count": int(confusion_matrix(y_test, test_predicted, labels=[0, 1])[0, 1])},
        {"actual_label": "Churned", "predicted_label": "Retained", "customer_count": int(confusion_matrix(y_test, test_predicted, labels=[0, 1])[1, 0])},
        {"actual_label": "Churned", "predicted_label": "Churned", "customer_count": int(confusion_matrix(y_test, test_predicted, labels=[0, 1])[1, 1])},
    ]
)

fact_roc_curve = pd.DataFrame(
    {
        "fpr": fpr,
        "tpr": tpr,
        "threshold": roc_thresholds,
    }
)

fact_precision_recall_curve = pd.DataFrame(
    {
        "precision": precision_curve,
        "recall": recall_curve,
        "threshold": pr_threshold_aligned,
    }
)

model_name = str(leaderboard.iloc[0]["Model"]) if "Model" in leaderboard.columns else type(best_model).__name__
dim_model_metrics = pd.DataFrame(
    [
        {
            "model_name": model_name,
            "auc": float(roc_auc_score(y_test, test_probability)),
            "recall": float(recall_score(y_test, test_predicted, zero_division=0)),
            "f1": float(f1_score(y_test, test_predicted, zero_division=0)),
            "precision": float(precision_score(y_test, test_predicted, zero_division=0)),
            "accuracy": float(accuracy_score(y_test, test_predicted)),
            "best_threshold": float(best_threshold_row["threshold"]),
            "best_threshold_f1": float(best_threshold_row["f1"]),
            "best_threshold_recall": float(best_threshold_row["recall"]),
            "best_threshold_precision": float(best_threshold_row["precision"]),
        }
    ]
)

full_scored = predict_model(best_model, data=X, raw_score=True).reset_index(drop=True)
full_score_column = _resolve_score_column(full_scored)
full_probability = pd.to_numeric(full_scored[full_score_column], errors="coerce").fillna(0.0).clip(0, 1)

customer_key = "CustomerID" if "CustomerID" in df_clean.columns else None
fact_customer_scores = pd.DataFrame(
    {
        "CustomerID": df_clean[customer_key].reset_index(drop=True) if customer_key else pd.Series(np.arange(1, len(df_clean) + 1)),
        "churn_probability": full_probability,
        "predicted_churn": (full_probability >= 0.50).astype(int),
        "risk_segment": pd.cut(
            full_probability,
            bins=[-0.01, 0.40, 0.70, 1.00],
            labels=["Low Risk", "Medium Risk", "High Risk"],
        ).astype(str),
    }
)

for column in ["Total_Spend", "Gender", "Contract_Length", "Subscription_Type", "Tenure"]:
    if column in df_clean.columns:
        fact_customer_scores[column] = df_clean[column].reset_index(drop=True)

dim_model_metrics.to_parquet(output_dir / "dim_model_metrics.parquet", index=False)
fact_confusion_matrix.to_parquet(output_dir / "fact_confusion_matrix.parquet", index=False)
fact_roc_curve.to_parquet(output_dir / "fact_roc_curve.parquet", index=False)
fact_precision_recall_curve.to_parquet(output_dir / "fact_precision_recall_curve.parquet", index=False)
fact_threshold_metrics.to_parquet(output_dir / "fact_threshold_metrics.parquet", index=False)
fact_customer_scores.to_parquet(output_dir / "fact_customer_scores.parquet", index=False)

print("Explainability parquet exports completed.")
display(dim_model_metrics)
display(fact_confusion_matrix)
display(fact_threshold_metrics.head())

Explainability parquet exports completed.


,model_name,auc,recall,f1,precision,accuracy,best_threshold,best_threshold_f1,best_threshold_recall,best_threshold_precision
0,Light Gradient Boosting Machine,0.954104,0.995648,0.943469,0.896486,0.93375,0.15,0.945083,0.99955,0.896245


,actual_label,predicted_label,customer_count
0,Retained,Retained,4571
1,Retained,Churned,766
2,Churned,Retained,29
3,Churned,Churned,6634


,threshold,precision,recall,f1,accuracy,predicted_churners
0,0.05,0.890970,0.999550,0.942142,0.931833,7475
1,0.10,0.895161,0.999550,0.944480,0.934750,7440
2,0.15,0.896245,0.999550,0.945083,0.935500,7431
3,0.20,0.896204,0.999100,0.944858,0.935250,7428
4,0.25,0.896389,0.998499,0.944693,0.935083,7422


## 7. Business Retention Insights
Use the best model + EDA signals to derive retention actions by segment, contract type, tenure, and payment behavior.